In [2]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
spark.conf.set("spark.sql.parquet.vorder.enabled", "true")
spark.conf.set("spark.microsoft.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.microsoft.delta.optimizeWrite.binSize", "1073741824")


StatementMeta(, 8ecaa549-9ba9-4d2e-87a2-997b54a0b0d0, 4, Finished, Available, Finished)

In [3]:
bronze_df = spark.read.table("dbo.bronze_weather_data")
bronze_df.show()

StatementMeta(, 8ecaa549-9ba9-4d2e-87a2-997b54a0b0d0, 5, Finished, Available, Finished)

+--------------------+-------+--------+--------------------+----------+-------+-----+--------+
|           Condition|Country|Humidity|            Location|PressureMB| Region|TempC|Wind_KPH|
+--------------------+-------+--------+--------------------+----------+-------+-----+--------+
|{1003, //cdn.weat...| Canada|      33|{Canada, 51.0833,...|    1017.0|Alberta|  7.0|     9.0|
|{1003, //cdn.weat...| Canada|      49|{Canada, 51.0833,...|    1014.0|Alberta|  8.1|     8.6|
|{1003, //cdn.weat...| Canada|      31|{Canada, 51.0833,...|    1017.0|Alberta|  8.2|     3.6|
|{1003, //cdn.weat...| Canada|      60|{Canada, 51.0833,...|    1018.0|Alberta|  2.2|     8.6|
|{1003, //cdn.weat...| Canada|      69|{Canada, 51.0833,...|    1017.0|Alberta|  1.1|     5.4|
|{1003, //cdn.weat...| Canada|      69|{Canada, 51.0833,...|    1010.0|Alberta|  1.2|     9.7|
|{1003, //cdn.weat...| Canada|      75|{Canada, 51.0833,...|    1020.0|Alberta|  2.3|     9.0|
|{1003, //cdn.weat...| Canada|      80|{Canada, 51

In [4]:
bronze_df.printSchema()

StatementMeta(, 8ecaa549-9ba9-4d2e-87a2-997b54a0b0d0, 6, Finished, Available, Finished)

root
 |-- Condition: struct (nullable = true)
 |    |-- code: long (nullable = true)
 |    |-- icon: string (nullable = true)
 |    |-- text: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Humidity: long (nullable = true)
 |-- Location: struct (nullable = true)
 |    |-- country: string (nullable = true)
 |    |-- lat: double (nullable = true)
 |    |-- localtime: string (nullable = true)
 |    |-- localtime_epoch: long (nullable = true)
 |    |-- lon: double (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- region: string (nullable = true)
 |    |-- tz_id: string (nullable = true)
 |-- PressureMB: double (nullable = true)
 |-- Region: string (nullable = true)
 |-- TempC: double (nullable = true)
 |-- Wind_KPH: double (nullable = true)



In [5]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

silver_df = bronze_df.select(
    F.col("Condition.code").alias("weather_key"),
    F.col("Condition.text").alias("weather_condition"),
    F.col("Country").alias("country"),
    F.col("Humidity").alias("humidity"),
    F.col("Location.lat").alias("latitude"),
    F.col("Location.lon").alias("longitude"),
    F.col("Location.name").alias("city"),
    F.col("Location.localtime").alias("localtime"),
    F.col("Location.tz_id").alias("tz_id"),
    F.col("PressureMB").alias("pressure_mb"),
    F.col("Region").alias("region"),
    F.col("TempC").alias("temperature_c"),
    F.col("Wind_KPH").alias("windkph"),


)\
 .withColumn("comfortindex", (F.col("humidity") + F.col("temperature_c"))/4) \
 .withColumn("date", F.to_date("localtime")) \
 .withColumn("month", F.month("localtime")) \
 .withColumn("year", F.year("localtime")) \
 .withColumn("day", F.day("localtime")) \
 .withColumn("hour_of_day", F.hour("localtime")) \
.withColumn("is_weekend", F.when(F.weekday(F.col("localtime")) >= 5, True).otherwise(False))


silver_table = "silver_weather_data"

silver_df.printSchema()

silver_table = "silver_weather_data"

silver_df.write \
.mode("overwrite") \
.format("delta") \
.option("overwriteSchema", "true") \
.saveAsTable(silver_table)

StatementMeta(, 8ecaa549-9ba9-4d2e-87a2-997b54a0b0d0, 7, Finished, Available, Finished)

root
 |-- weather_key: long (nullable = true)
 |-- weather_condition: string (nullable = true)
 |-- country: string (nullable = true)
 |-- humidity: long (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- city: string (nullable = true)
 |-- localtime: string (nullable = true)
 |-- tz_id: string (nullable = true)
 |-- pressure_mb: double (nullable = true)
 |-- region: string (nullable = true)
 |-- temperature_c: double (nullable = true)
 |-- windkph: double (nullable = true)
 |-- comfortindex: double (nullable = true)
 |-- date: date (nullable = true)
 |-- month: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- hour_of_day: integer (nullable = true)
 |-- is_weekend: boolean (nullable = false)



In [6]:
display(silver_df)

StatementMeta(, 8ecaa549-9ba9-4d2e-87a2-997b54a0b0d0, 8, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, ad90ddf8-f3e9-4adb-a107-dbb18af009b0)